In [1]:
#Copyright 2026 Mücahit Sahin
#
#Licensed under the Apache License, Version 2.0 (the "License");
#you may not use this file except in compliance with the License.
#You may obtain a copy of the License at
#
#    http://www.apache.org/licenses/LICENSE-2.0
#
#Unless required by applicable law or agreed to in writing, software
#distributed under the License is distributed on an "AS IS" BASIS,
#WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
#See the License for the specific language governing permissions and
#limitations under the License.

In [3]:
import pandas as pd

from utils import gpt
from utils import feature_generator

In [4]:
all_datasets = pd.read_csv("data/Benchmark-Labeled-Data/all_data.csv")
all_datasets.Attribute_name = all_datasets.Attribute_name.astype('str') # NaNs
all_datasets.head()

,Record_id,Attribute_name,y_act,total_vals,num_nans,%_nans,num_of_dist_val,%_dist_val,mean,std_dev,...,mean_stopword_total,stdev_stopword_total,mean_char_count,stdev_char_count,mean_whitespace_count,stdev_whitespace_count,mean_delim_count,stdev_delim_count,is_list,is_long_sentence
0,33,Area,categorical,21477,0,0.0,174,0.810169,0.000000,0.000000,...,0.2,0.4,10.0,4.816638,0.4,0.8,0.4,0.8,False,False
1,33,Area Code,categorical,21477,0,0.0,174,0.810169,125.449411,72.866452,...,0.0,0.0,1.0,0.000000,0.0,0.0,0.0,0.0,False,False
2,33,Element,categorical,21477,0,0.0,2,0.009312,0.000000,0.000000,...,0.0,0.0,4.0,0.000000,0.0,0.0,0.0,0.0,False,False
3,33,Element Code,categorical,21477,0,0.0,2,0.009312,5211.687154,146.816661,...,0.0,0.0,4.0,0.000000,0.0,0.0,0.0,0.0,False,False
4,33,Item,categorical,21477,0,0.0,115,0.535457,0.000000,0.000000,...,0.8,0.4,19.6,2.244994,2.0,0.0,2.0,0.0,False,False


# Define LLM Prompts

In [5]:
def build_feature_value_list(row):
    feature_values = []

    for i in range(1, 6):
        v = row.loc[0, f'sample_{i}']

        try:
            parsed = int(v)
        except:
            try:
                parsed = float(str(v))
            except:
                # if len(str(v)) > 200:
                #     parsed = str(v)[:200]  # truncate strings
                #     parsed = parsed + ' ...'
                # else:
                #     parsed = str(v)

                tokenizer = tiktoken.get_encoding("cl100k_base")
                text = str(v)
                tokens = tokenizer.encode(text)

                if len(tokens) > 150:
                    tokens = tokens[:150]
                    text = tokenizer.decode(tokens)
                
                parsed = text

        feature_values.append(parsed)

    return feature_values


def LLM_generate_base_prompt(row):
    feature_name = row.loc[0, 'Attribute_name']
    nr_distinct_values = row.loc[0, 'num_of_dist_val']
    nr_total_values = row.loc[0, 'total_vals']

    feature_values = build_feature_value_list(row)

    # Build human-readable list with sample labels
    feature_values_str = ""
    for i, v in enumerate(feature_values, start=1):
        if isinstance(v, str):
            v_str = f'"{v}"'
        else:
            v_str = str(v)
        feature_values_str += f"## Sample {i} ##\n{v_str}\n"
        if not i == len(feature_values):
            feature_values_str += "\n"

    base_prompt = (
        f"I have a dataset which contains a feature with the name '{feature_name}'."
        f" The dataset includes {nr_total_values} values in total."
        f" The feature has {nr_distinct_values} unique values in total."
        f" Here are five distinct sample values from this feature:\n\n"
        f"```\n{feature_values_str}\n```\n"
        f"\n**Task**:\n"
    )

    return base_prompt

In [6]:
def get_fewshot_indexes(n, dataset, label, label_col, seed, class_map=None):
    examples = []
    if label.lower() == 'all':
        if not class_map:
            return 'Please also provide the class map!'
        else:
            for c in class_map:
                true_labels = dataset.loc[(dataset[label_col]==class_map[c])].sample(n, random_state=seed).index.tolist()
                for i in range(n):
                    examples.append(true_labels[i])
    else:
        true_labels = dataset.loc[(dataset[label_col]==label)].sample(n, random_state=seed).index.tolist()
        false_labels = dataset.loc[~(dataset[label_col]==label)].sample(n, random_state=seed).index.tolist()
        
        for i in range(n):
            examples.append(true_labels[i])
            examples.append(false_labels[i])

    return examples

In [7]:
def generate_fewshot_examples(class_prompt, dataset, indexes, binary=False):
    prompt = f"\n{25*'/'}"
    prompt += f" WHEN ANSWERING USER QUESTIONS FOLLOW THESE EXAMPLES {25*'/'}\n\n\n"
    for n, i in enumerate(indexes):
        prompt += f"**Question_{n+1}**:\n"
        prompt += LLM_generate_base_prompt(dataset.loc[i:i+1].reset_index(drop=True))
        prompt = prompt.replace('**Task**:', f'**Task_{n+1}**:')
        prompt += class_prompt
        label = dataset.loc[i:i+1].y_act.values[0].title()
        if binary:
            if n % 2 == 0:
                label = 'yes'
            else:
                label = 'no'
        prompt += f"\n\n\n**Answer_{n+1}**:\n{label}\n\n"
        if not n == len(indexes)-1:
            prompt += f"{25*'~'}\n\n"
    return prompt

## System Prompt

In [8]:
system_prompt = (
    "You are a Data Scientist tasked with accurately classifying features into 9 predefined classes: "
    "Numeric, Categorical, Datetime, Sentence, URL, List, Embedded-Number, Not-Generalizable, and Context-Specific. "
    "For each feature, your objective is to:\n\n"
    "1) Analyze the feature name, the sample values, and the summary statistics provided.\n"
    "2) Assign the feature to the most specific and accurate class based on predefined criteria.\n\n"
    "Ensure that you:\n"
    "- Use precise and confident language for classification.\n"
    "- Assign a class only when there is sufficient evidence to support the decision.\n"
    "- Avoid ambiguity and overly broad classifications.\n\n"
    "Below are detailed descriptions of each class to guide your decision-making:\n\n"
    "1. Numeric:\n"
    "This class includes features containing raw numerical values, such as integers, floating-point numbers, or negative numbers "
    "(e.g., '-5', '12.34'). These features should represent measurable quantities or counts and are usually continuous or discrete. "
    "Numeric features are suitable for mathematical operations like addition, subtraction, multiplication, and division, and are used "
    "to model and predict numeric outcomes. Common examples include 'age', 'price', 'distance', or 'sales volume'.\n"
    "Important Notes for Classifying Numeric Features:\n"
    " - Numeric features that use alternative characters for decimal separators (e.g., commas) are classified as Embedded-Number rather than Numeric.\n"
    " - Numeric features should be continuous or discrete quantities with a meaningful numeric relationship, e.g., 'temperature', "
    "'income', or 'height'.\n"
    " - Features with a small number of distinct numeric values, such as binary features ('0' and '1'), should be classified as numeric "
    "only if they represent actual quantities, not categories.\n"
    " - Features used for classification purposes (e.g., '0' for 'No' and '1' for 'Yes') should be classified as Categorical, even "
    "if their values are numeric.\n"
    " - Avoid classifying identifiers or domain-specific codes (e.g., 'ProductID', 'ZipCode') as Numeric, even if they contain "
    "numeric values.\n"
    "Additional Guidance:\n"
    " - Ensure the numeric feature represents a measurable quantity and can undergo statistical or machine learning analysis as "
    "a continuous or discrete quantity.\n\n"
    "2. Categorical:\n"
    "Categorical features contain distinct labels or group identifiers, such as product names, tags, or classifications. Examples "
    "include 'red', 'apple', 'high', 'low', or 'yes'. Categorical features may include numeric data representing encoded categories "
    "(e.g., one-hot or label encoding).\n"
    "Important Notes for Classifying Categorical Features:\n"
    " - Categorical features represent groups or labels rather than quantities or numbers.\n"
    " - Numeric data that encodes categories (e.g., '1' for 'Yes', '0' for 'No') should be classified as Categorical, not Numeric.\n"
    " - Avoid classifying features like dates, URLs, or sentences as Categorical.\n"
    " - Features that are lists of categories (e.g., ['apple', 'banana', 'orange']) should not be classified here.\n" 
    "Additional Guidance:\n"
    " - Ensure the categorical feature consists of distinct groups or categories and does not contain continuous or ordinal "
    "numeric values.\n\n"
    "3. Datetime:\n"
    "Datetime features represent date or time information, such as 'March 2020', '15-Mar-2020', or '2025-01-01'. These features "
    "can include partial date formats (e.g., year and month) but should not be used to represent a single point in time like '2020' "
    "without additional context.\n"
    "Important Notes for Classifying Datetime Features:\n"
    " - The feature should represent time in some format (e.g., date, timestamp, year, month, day).\n"
    " - Partial dates (e.g., year and month) can be classified as datetime but exclude year-only formats.\n"
    "Additional Guidance:\n"
    " - Ensure the datetime feature contains a temporal component and is not just an identifier or code.\n\n"
    "4. Sentence:\n"
    "Sentence features represent natural language text conveying meaning, context, or information in the form of complete sentences "
    "or phrases. Short entries (1-3 words) may also be classified as sentences if they express meaningful ideas, e.g., 'Great job!' "
    "or 'Red sedan.'\n"
    "Important Notes for Classifying Sentence Features:\n"
    " - The feature should consist of natural language text that conveys a meaningful idea.\n"
    " - Avoid classifying text as a sentence if it functions purely as a label, identifier, or category (e.g., 'NY' or 'Electronics').\n"
    "Additional Guidance:\n"
    " - Ensure the sentence feature contains a meaningful message or statement.\n\n"
    "5. Embedded-Number:\n"
    "This class includes features with numbers embedded alongside special characters or units, such as 'USD 100' or '5,000 MHz'. "
    "These features may also include other symbols like currency or percentage signs. "
    "Numerical values that use alternative decimal separators, such as commas (e.g., 53,12), regardless if they are used with a unit or not, also belong to this class.\n"
    "Important Notes for Classifying Embedded-Number Features:\n"
    " - The feature should include numeric values along with a unit, symbol, or special character.\n"
    " - Numbers that use alternative characters for decimal or thousand separators (e.g., commas) belong to this class.\n"
    " - Features that are lists of numbers (e.g., '[1, 2, 3]') should not be classified here.\n"
    "Additional Guidance:\n"
    " - Ensure the numeric value and unit/symbol together represent a measurable quantity, such as '10 USD' or '5,000 MHz'.\n\n"
    "6. URL:\n"
    "Features in this class represent web addresses, parts of web addresses, or related components. Examples include full URLs "
    "('https://example.com'), domain names ('example.com'), or URL fragments ('/products/item'). These features typically include "
    "components such as protocols ('http://', 'https://'), domains ('example.com'), paths ('/path/to/resource'), or query strings "
    "('?id=123').\n"
    "Important Notes for Classifying URL Features:\n"
    " - The feature should represent an internet address or network resource.\n"
    " - Avoid classifying features with ambiguous or inconsistent URL formats.\n"
    "Additional Guidance:\n"
    " - Ensure the feature is an actual URL or related network address, not just a part of a URL or a reference.\n\n"
    "7. List:\n"
    "This class includes features whose values represent lists, where each value contains multiple discrete items or elements. "
    "These items may be integers, strings, or any other data type, and they should be in a structured format such as a list of items "
    "separated by commas, or enclosed in square brackets like '[apple, banana, cherry]'. A key distinction is that the values should explicitly "
    "represent multiple items or objects in a list format, not just a string of text with commas.\n"
    "Important Notes for Classifying List Features:\n"
    " - A List feature should contain multiple distinct elements (e.g., ['cat', 'dog', 'bird']).\n"
    " - If the values are comma-separated items within a string, they must represent individual objects or entities in a list structure, "
    "such as a list of keywords, items, or other discrete data points.\n"
    " - Avoid classifying long sentences or descriptive text as a List, even if there are commas present in the value. "
    "Features like 'Reckless driving, Poor choice in music' represent sentences, not lists, because the items are not clearly separated "
    "as discrete list elements.\n"
    " - Lists must clearly indicate multiple distinct elements and follow a recognizable list format.\n\n"
    "8. Not-Generalizable:\n"
    "A feature is classified as 'Not-Generalizable' if it has little to no predictive value. This includes unique identifiers "
    "like 'CustomerID' or features with only one unique value across the entire dataset (e.g., all 'NA').\n"
    "Important Notes for Classifying Not-Generalizable Features:\n"
    " - Features with unique identifiers or nearly constant values should be classified as Not-Generalizable.\n"
    " - Do not classify features with little variance as belonging to other classes like Numeric or Categorical.\n"
    "Additional Guidance:\n"
    " - Ensure that the feature provides sufficient variation to be useful in modeling or analysis.\n\n"
    "9. Context-Specific:\n"
    "Features in this class require additional domain knowledge to classify, such as industry-specific data, ambiguous formats, "
    "or specialized data types.\n"
    "Important Notes for Classifying Context-Specific Features:\n"
    " - The feature's role or meaning cannot be determined without understanding the specific context or domain.\n"
    " - Features with ambiguous names or formats may belong in this class.\n"
    "Additional Guidance:\n"
    " - Use Context-Specific when no other predefined class confidently fits the feature.\n"
    " - Avoid classifying features with standard formats (Numeric, Categorical, Datetime) as Context-Specific.\n"
    "Decision hierarchy:\n"
    "1. Prioritize matches to numeric, categorical, datetime, or other clear types when evidence aligns strongly.\n"
    "2. Use Context-Specific as a fallback for ambiguous, specialized, or complex features requiring domain knowledge.\n\n"
    "Your goal is to ensure accurate, reliable, and meaningful classifications for all features."
)

# GPT-5.4

In [9]:
def GPT_5_4_detect_all_at_once(row, as_batch=False):
    prompt = LLM_generate_base_prompt(row) + \
    f"Based on the provided information," + \
    f" classify the feature as either Numeric, Categorical, Datetime, Sentence, URL, List, Embedded-Number," + \
    f" Not-Generalizable, or Context-Specific." + \
    f" Respond with ONLY the class name in lower case."
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_detect_all_at_once_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        return response.lower()


def GPT_5_4_is_numeric(row, as_batch=False):
    prompt = LLM_generate_base_prompt(row) + \
    f"Based on the provided information," + \
    f" can you classify the feature as Numeric?" + \
    f" Respond with ONLY 'yes' or 'no'."
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_numeric_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_categorical(row, as_batch=False):
    prompt = LLM_generate_base_prompt(row) + \
    f"Based on the provided information," + \
    f" can you classify the feature as Categorical?" + \
    f" Respond with ONLY 'yes' or 'no'."
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_categorical_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_datetime(row, as_batch=False):
    prompt = LLM_generate_base_prompt(row) + \
    f"Based on the provided information," + \
    f" can you classify the feature as Datetime?" + \
    f" Respond with ONLY 'yes' or 'no'."
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_datetime_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_sentence(row, as_batch=False):
    prompt = LLM_generate_base_prompt(row) + \
    f"Based on the provided information," + \
    f" can you classify the feature as Sentence?" + \
    f" Respond with ONLY 'yes' or 'no'."
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_sentence_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_embedded_number(row, as_batch=False):
    prompt = LLM_generate_base_prompt(row) + \
    f"Based on the provided information," + \
    f" can you classify the feature as Embedded-Number?" + \
    f" Respond with ONLY 'yes' or 'no'."
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_embedded_number_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_url(row, as_batch=False):
    prompt = LLM_generate_base_prompt(row) + \
    f"Based on the provided information," + \
    f" can you classify the feature as URL?" + \
    f" Respond with ONLY 'yes' or 'no'."
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_url_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_list(row, as_batch=False):
    prompt = LLM_generate_base_prompt(row) + \
    f"Based on the provided information," + \
    f" can you classify the feature as List?" + \
    f" Respond with ONLY 'yes' or 'no'."
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_list_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_not_generalizable(row, as_batch=False):
    prompt = LLM_generate_base_prompt(row) + \
    f"Based on the provided information," + \
    f" can you classify the feature as Not-Generalizable?" + \
    f" Respond with ONLY 'yes' or 'no'."
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_not_generalizable_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_context_specific(row, as_batch=False):
    prompt = LLM_generate_base_prompt(row) + \
    f"Based on the provided information," + \
    f" can you classify the feature as Context-Specific?" + \
    f" Respond with ONLY 'yes' or 'no'."
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_context_specific_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'

## Few-Shot

In [10]:
def GPT_5_4_detect_all_at_once_fewshot(row, as_batch=False):
    class_prompt = f"Based on the provided information," + \
    f" classify the feature as either Numeric, Categorical, Datetime, Sentence, URL, List, Embedded-Number," + \
    f" Not-Generalizable, or Context-Specific." + \
    f" Respond with ONLY the class name in lower case."
    fewshot_examples_indexes = [780, 4827, 6903, 7397, 7684, 7339, 7243, 5438, 5559]
    fewshot_examples = generate_fewshot_examples(class_prompt, all_datasets, fewshot_examples_indexes)
    system_prompt_fewshot = system_prompt + '\n\n' + fewshot_examples
    prompt = LLM_generate_base_prompt(row) + class_prompt

    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_detect_all_at_once_fewshot_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt_fewshot},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        return response.lower()


def GPT_5_4_is_numeric_fewshot(row, as_batch=False):
    class_prompt = f"Based on the provided information," + \
    f" can you classify the feature as Numeric?" + \
    f" Respond with ONLY 'yes' or 'no'."
    fewshot_examples_indexes = [380, 6775, 509, 7373, 3153, 7853, 3801, 4071, 761, 5743]
    fewshot_examples = generate_fewshot_examples(class_prompt, all_datasets, fewshot_examples_indexes, binary=True)
    system_prompt_fewshot = system_prompt + '\n\n' + fewshot_examples
    prompt = LLM_generate_base_prompt(row) + class_prompt

    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_numeric_fewshot_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt_fewshot},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_categorical_fewshot(row, as_batch=False):
    class_prompt = f"Based on the provided information," + \
    f" can you classify the feature as Categorical?" + \
    f" Respond with ONLY 'yes' or 'no'."
    fewshot_examples_indexes = [4980, 9227, 6513, 9126, 2795, 2900, 1142, 3540, 3925, 9533]
    fewshot_examples = generate_fewshot_examples(class_prompt, all_datasets, fewshot_examples_indexes, binary=True)
    system_prompt_fewshot = system_prompt + '\n\n' + fewshot_examples
    prompt = LLM_generate_base_prompt(row) + class_prompt
    
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_categorical_fewshot_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt_fewshot},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_datetime_fewshot(row, as_batch=False):
    class_prompt = f"Based on the provided information," + \
    f" can you classify the feature as Datetime?" + \
    f" Respond with ONLY 'yes' or 'no'."
    fewshot_examples_indexes = [7619, 1314, 3104, 3445, 6970, 8186, 6892, 3787, 3136, 5666]
    fewshot_examples = generate_fewshot_examples(class_prompt, all_datasets, fewshot_examples_indexes, binary=True)
    system_prompt_fewshot = system_prompt + '\n\n' + fewshot_examples
    prompt = LLM_generate_base_prompt(row) + class_prompt
    
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_datetime_fewshot_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt_fewshot},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_sentence_fewshot(row, as_batch=False):
    class_prompt = f"Based on the provided information," + \
    f" can you classify the feature as Sentence?" + \
    f" Respond with ONLY 'yes' or 'no'."
    fewshot_examples_indexes = [874, 944, 3536, 3091, 7933, 8409, 6872, 9074, 3333, 5930]
    fewshot_examples = generate_fewshot_examples(class_prompt, all_datasets, fewshot_examples_indexes, binary=True)
    system_prompt_fewshot = system_prompt + '\n\n' + fewshot_examples
    prompt = LLM_generate_base_prompt(row) + class_prompt
    
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_sentence_fewshot_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt_fewshot},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_embedded_number_fewshot(row, as_batch=False):
    class_prompt = f"Based on the provided information," + \
    f" can you classify the feature as Embedded-Number?" + \
    f" Respond with ONLY 'yes' or 'no'."
    fewshot_examples_indexes = [7730, 6352, 7186, 6588, 7220, 9839, 6778, 5192, 7926, 1908]
    fewshot_examples = generate_fewshot_examples(class_prompt, all_datasets, fewshot_examples_indexes, binary=True)
    system_prompt_fewshot = system_prompt + '\n\n' + fewshot_examples
    prompt = LLM_generate_base_prompt(row) + class_prompt
    
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_embedded_number_fewshot_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt_fewshot},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_url_fewshot(row, as_batch=False):
    class_prompt = f"Based on the provided information," + \
    f" can you classify the feature as URL?" + \
    f" Respond with ONLY 'yes' or 'no'."
    fewshot_examples_indexes = [7677, 309, 7612, 2112, 7105, 9899, 7689, 5677, 7648, 5525]
    fewshot_examples = generate_fewshot_examples(class_prompt, all_datasets, fewshot_examples_indexes, binary=True)
    system_prompt_fewshot = system_prompt + '\n\n' + fewshot_examples
    prompt = LLM_generate_base_prompt(row) + class_prompt
    
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_url_fewshot_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt_fewshot},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_list_fewshot(row, as_batch=False):
    class_prompt = f"Based on the provided information," + \
    f" can you classify the feature as List?" + \
    f" Respond with ONLY 'yes' or 'no'."
    fewshot_examples_indexes = [7577, 8439, 9353, 5786, 9354, 3217, 7538, 7960, 9763, 5310]
    fewshot_examples = generate_fewshot_examples(class_prompt, all_datasets, fewshot_examples_indexes, binary=True)
    system_prompt_fewshot = system_prompt + '\n\n' + fewshot_examples
    prompt = LLM_generate_base_prompt(row) + class_prompt
    
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_list_fewshot_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt_fewshot},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_not_generalizable_fewshot(row, as_batch=False):
    class_prompt = f"Based on the provided information," + \
    f" can you classify the feature as Not-Generalizable?" + \
    f" Respond with ONLY 'yes' or 'no'."
    fewshot_examples_indexes = [4607, 9120, 8179, 8837, 394, 2416, 4408, 4380, 5844, 4501]
    fewshot_examples = generate_fewshot_examples(class_prompt, all_datasets, fewshot_examples_indexes, binary=True)
    system_prompt_fewshot = system_prompt + '\n\n' + fewshot_examples
    prompt = LLM_generate_base_prompt(row) + class_prompt
    
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_not_generalizable_fewshot_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt_fewshot},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'


def GPT_5_4_is_context_specific_fewshot(row, as_batch=False):
    class_prompt = f"Based on the provided information," + \
    f" can you classify the feature as Context-Specific?" + \
    f" Respond with ONLY 'yes' or 'no'."
    fewshot_examples_indexes = [5559, 2183, 9386, 100, 1582, 2856, 5686, 3397, 104, 6491]
    fewshot_examples = generate_fewshot_examples(class_prompt, all_datasets, fewshot_examples_indexes, binary=True)
    system_prompt_fewshot = system_prompt + '\n\n' + fewshot_examples
    prompt = LLM_generate_base_prompt(row) + class_prompt
    
    if as_batch:
        gpt.add_batch(batchfile='batches/GPT_5_4_is_context_specific_fewshot_batch.jsonl',
                      model='gpt-5.4',
                      messages=[{"role": "system", "content": system_prompt_fewshot},
                                {"role": "user", "content": prompt}])
    else:
        response = gpt.ask_gpt(prompt, system_prompt, model="gpt-4o")
        if "yes" in response.lower():
            return True
        elif "no" in response.lower():
            return False
        else:
            return 'Failed to make LLM-prediction'

# LLM Features

In [11]:
features = {
    # GPT-5.4
    # Zero-Shot
    0: GPT_5_4_detect_all_at_once,
    1: GPT_5_4_is_numeric,
    2: GPT_5_4_is_categorical,
    3: GPT_5_4_is_datetime,
    4: GPT_5_4_is_sentence,
    5: GPT_5_4_is_url,
    6: GPT_5_4_is_embedded_number,
    7: GPT_5_4_is_list,
    8: GPT_5_4_is_not_generalizable,
    9: GPT_5_4_is_context_specific,

    # GPT-5.4
    # Few-Shot
    10: GPT_5_4_detect_all_at_once_fewshot,
    11: GPT_5_4_is_numeric_fewshot,
    12: GPT_5_4_is_categorical_fewshot,
    13: GPT_5_4_is_datetime_fewshot,
    14: GPT_5_4_is_sentence_fewshot,
    15: GPT_5_4_is_url_fewshot,
    16: GPT_5_4_is_embedded_number_fewshot,
    17: GPT_5_4_is_list_fewshot,
    18: GPT_5_4_is_not_generalizable_fewshot,
    19: GPT_5_4_is_context_specific_fewshot
}

# Gather results of the new features and extend the base feature set

In [12]:
base_features = pd.read_csv("data/Benchmark-Labeled-Data/all_data.csv")
base_features.head()

,Record_id,Attribute_name,y_act,total_vals,num_nans,%_nans,num_of_dist_val,%_dist_val,mean,std_dev,...,mean_stopword_total,stdev_stopword_total,mean_char_count,stdev_char_count,mean_whitespace_count,stdev_whitespace_count,mean_delim_count,stdev_delim_count,is_list,is_long_sentence
0,33,Area,categorical,21477,0,0.0,174,0.810169,0.000000,0.000000,...,0.2,0.4,10.0,4.816638,0.4,0.8,0.4,0.8,False,False
1,33,Area Code,categorical,21477,0,0.0,174,0.810169,125.449411,72.866452,...,0.0,0.0,1.0,0.000000,0.0,0.0,0.0,0.0,False,False
2,33,Element,categorical,21477,0,0.0,2,0.009312,0.000000,0.000000,...,0.0,0.0,4.0,0.000000,0.0,0.0,0.0,0.0,False,False
3,33,Element Code,categorical,21477,0,0.0,2,0.009312,5211.687154,146.816661,...,0.0,0.0,4.0,0.000000,0.0,0.0,0.0,0.0,False,False
4,33,Item,categorical,21477,0,0.0,115,0.535457,0.000000,0.000000,...,0.8,0.4,19.6,2.244994,2.0,0.0,2.0,0.0,False,False


In [13]:
gpt.load_api_key()

In [14]:
df = pd.read_csv('results/featurized_data.csv')

In [15]:
list(df.columns)[1478:]

[]

In [16]:
len(df)

9921

In [16]:
# Create batches for GPT-5.4
for i in range(0, 20):
    feature_generator.create_batchfiles(df=base_features, feature=features[i])

Batch for feature 'GPT_5_4_detect_all_at_once' added.
Make sure now to run the script /batches/run_llm.sh to retrieve batch results!

Batch for feature 'GPT_5_4_is_numeric' added.
Make sure now to run the script /batches/run_llm.sh to retrieve batch results!

Batch for feature 'GPT_5_4_is_categorical' added.
Make sure now to run the script /batches/run_llm.sh to retrieve batch results!

Batch for feature 'GPT_5_4_is_datetime' added.
Make sure now to run the script /batches/run_llm.sh to retrieve batch results!

Batch for feature 'GPT_5_4_is_sentence' added.
Make sure now to run the script /batches/run_llm.sh to retrieve batch results!

Batch for feature 'GPT_5_4_is_url' added.
Make sure now to run the script /batches/run_llm.sh to retrieve batch results!

Batch for feature 'GPT_5_4_is_embedded_number' added.
Make sure now to run the script /batches/run_llm.sh to retrieve batch results!

Batch for feature 'GPT_5_4_is_list' added.
Make sure now to run the script /batches/run_llm.sh to re

In [17]:
################ RUN GPT-5.4 BATCHING EXPERIMENTS ##################
######### ---> LONG WAITING TIMES BECAUSE OF BATCHING <--- #########

!cd batches && ./run_llm.sh

Parallel LLM feature processing
Max parallel jobs: 3
Batch-IDs are stored in file: batches/batch_ids.txt
▶ Starting: GPT_5_4_is_numeric
> To check the current status of the running batch, look into the file: ../logs/GPT_5_4_is_numeric.log
▶ Starting: GPT_5_4_is_categorical
> To check the current status of the running batch, look into the file: ../logs/GPT_5_4_is_categorical.log
Completed: GPT_5_4_is_categorical
Completed: GPT_5_4_is_numeric
All features finished


In [ ]:
for i in range(0, 20):
    featurized_data = feature_generator.get_feature_results(df=base_features,
                                                            feature=features[i],
                                                            as_batch=True,
                                                            save_path='results/featurized_data_with_gpt_features.csv')

In [ ]:
featurized_data.head()